![walmartecomm](data/walmartecomm.jpg)

Walmart is the biggest retail store in the United States. Just like others, they have been expanding their e-commerce part of the business. By the end of 2022, e-commerce represented a roaring $80 billion in sales, which is 13% of total sales of Walmart. One of the main factors that affects their sales is public holidays, like the Super Bowl, Labour Day, Thanksgiving, and Christmas. 

In this project, the task is to create a data pipeline for the analysis of supply and demand around the holidays, along with conducting a preliminary analysis of the data. There are two data sources: grocery sales and complementary data. `grocery_sales` table in `PostgreSQL` database with the following features:

# `grocery_sales`
- `"index"` - unique ID of the row
- `"Store_ID"` - the store number
- `"Date"` - the week of sales
- `"Weekly_Sales"` - sales for the given store

Also, the `extra_data.parquet` file that contains complementary data:

# `extra_data.parquet`
- `"IsHoliday"` - Whether the week contains a public holiday - 1 if yes, 0 if no.
- `"Temperature"` - Temperature on the day of sale
- `"Fuel_Price"` - Cost of fuel in the region
- `"CPI"` – Prevailing consumer price index
- `"Unemployment"` - The prevailing unemployment rate
- `"MarkDown1"`, `"MarkDown2"`, `"MarkDown3"`, `"MarkDown4"` - number of promotional markdowns
- `"Dept"` - Department Number in each store
- `"Size"` - size of the store
- `"Type"` - type of the store (depends on `Size` column)

Merge those files and perform some data manipulations. The transformed DataFrame can then be stored as the `clean_data` variable containing the following columns:
- `"Store_ID"`
- `"Month"`
- `"Dept"`
- `"IsHoliday"`
- `"Weekly_Sales"`
- `"CPI"`
- "`"Unemployment"`"

After merging and cleaning the data, analyze monthly sales of Walmart and store the results of the analysis as the `agg_data` variable that should look like:

|  Month | Weekly_Sales  | 
|---|---|
| 1.0  |  33174.178494 |
|  2.0 |  34333.326579 |
|  ... | ...  |  

Finally, save the `clean_data` and `agg_data` as the csv files.

It is recommended to use `pandas` for this project. 

In [37]:
# import lib
import pandas as pd
import os

In [38]:
# Extract function
def extract(store_data, extra_data):
    extra_df = pd.read_parquet(extra_data)
    merged_df = store_data.merge(extra_df, on = "index")
    return merged_df

In [39]:
# Create the transform() function with one parameter: "raw_data"
def transform(raw_data):
    # step 1 - identify fields with numeric & fill with mean
    num_cols = raw_data.select_dtypes(include='number').columns
    raw_data[num_cols] = raw_data[num_cols].fillna(raw_data[num_cols].mean())

    # step 2 - adding a column month & keep only weekly sales over 10,000
    raw_data['Date'] = pd.to_datetime(raw_data['Date'], errors='coerce')
    raw_data['Month'] = raw_data['Date'].dt.month
    raw_data = raw_data[raw_data['Weekly_Sales'] > 10000] 
    raw_data = raw_data.drop(columns=['index', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'Type', 'Size','Date'])
    return raw_data

In [40]:
# Create the avg_weekly_sales_per_month function that takes in the cleaned data from the last step
def avg_weekly_sales_per_month(clean_data):
    df = clean_data.loc[:,['Month','Weekly_Sales']]
    df = df.groupby('Month').agg('mean').reset_index().round(2)
    df = df.rename(columns={'Weekly_Sales':'Avg_Sales'})
    return df

In [41]:
# Create the load() function that takes in the cleaned DataFrame and the aggregated one with the paths where they are going to be stored
def load(full_data, full_data_file_path, agg_data, agg_data_file_path):
    full_data.to_csv(full_data_file_path, index=False)
    agg_data.to_csv(agg_data_file_path, index=False)

In [42]:
# Create the validation() function with one parameter: file_path - to check whether the previous function was correctly executed
def validation(*file_path):
    missing = [f for f in file_path if not os.path.exists(f)]
    if missing:
        raise FileNotFoundError(f"Missing files: {missing}")
    return True

In [43]:
# load data - Extracted from SQL
grocery_sales = pd.read_csv('data/grocery_sales.csv')

# Call the extract() function and store it as the "merged_df" variable
merged_df = extract(grocery_sales, "data/extra_data.parquet")

# Call the transform() function and pass the merged DataFrame
clean_data = transform(merged_df)

# Call the avg_weekly_sales_per_month() function and pass the cleaned DataFrame
agg_data = avg_weekly_sales_per_month(clean_data)

# print agg data
print(agg_data)

# Call the load() function and pass the cleaned and aggregated DataFrames with their paths    
load(clean_data, 'clean_data.csv', agg_data, 'agg_data.csv')

    Month  Avg_Sales
0     1.0   33174.18
1     2.0   34333.33
2     3.0   33220.89
3     4.0   33392.37
4     5.0   33339.89
5     6.0   34582.47
6     7.0   33922.76
7     8.0   33644.79
8     9.0   33258.05
9    10.0   32736.99
10   11.0   36594.03
11   12.0   39238.80


In [44]:
# Call the validation() function and pass first, the cleaned DataFrame path, and then the aggregated DataFrame path
validation('clean_data.csv', 'agg_data.csv')

True